In [2]:
import os
from dotenv  import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq

In [3]:
load_dotenv()
MODEL=os.getenv("GROQ_MODEL","openai/gpt-oss-20b")
llm=ChatGroq(model=MODEL,temperature=0.2)


In [4]:
# Functions defining, it can be anyhting 
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Faked here on purpose — swap in a real API call (e.g. Open-Meteo, which is free
    # and needs no key) once you're comfortable with the pattern.
    fake_data = {"hyderabad": "32°C, humid", "bengaluru": "24°C, cloudy"}
    return fake_data.get(city.lower(), f"No data for {city}, assume 28°C and sunny.")


def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

In [5]:
agent = create_agent(

    model=llm,
    tools=[get_weather,add_numbers],
    system_prompt=("You are a helpful assistant. Use the available tools when they "
            "would give a more accurate answer than guessing."
        ),

)

In [19]:
if __name__ == "__main__":
    result=agent.invoke(
        {"messages": [{"role": "user","content": "What's the weather in Hyderabad, and what's 42 + 58?" }]})
     # `create_agent` returns the full conversation state — the last message is the answer.
    final_message = result["messages"][-1]
    print("Final answer:", final_message.content)
    print("\n--- Full trace of messages (see how the agent called tools) ---")
    for m in result["messages"]:
        print(f"[{m.type}] {m.content if m.content else m.tool_calls}")
    # for m in result["messages"]:
    #     print(f"[{m.type}] {m.content if m.content else m.tool_calls}")
    

Final answer: **Hyderabad Weather:** 32 °C, humid  
**42 + 58:** 100.0

--- Full trace of messages (see how the agent called tools) ---
[human] What's the weather in Hyderabad, and what's 42 + 58?
[ai] [{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': 'fc_880bc60d-2cc6-46bd-940b-7675f0bba6c2', 'type': 'tool_call'}]
[tool] 32°C, humid
[ai] [{'name': 'add_numbers', 'args': {'a': 42, 'b': 58}, 'id': 'fc_18cc6349-661a-4507-ade5-fdd204948422', 'type': 'tool_call'}]
[tool] 100.0
[ai] **Hyderabad Weather:** 32 °C, humid  
**42 + 58:** 100.0
